# **CCDS – SentinelMind (Cognitive Cyber Defense System with Human Behavior Intelligence)**
SentinelMind (CCDS v4.3) is an advanced AI-powered Cognitive Cyber Defense System that detects cyber threats by analyzing human behavior instead of only system logs.

The system continuously monitors:

Facial micro-expressions (webcam)

Voice stress patterns (microphone)

Behavioral anomalies (typing & interaction behavior)

Emotional instability using deep learning

It combines Computer Vision, Audio AI, Behavioral AI, and 3D Interactive Forensics to calculate a real-time Threat Risk Score, visualize it in interactive 3D graphs, and explain the reason behind each alert.

This approach enables early threat detection, including insider threats, coercion, social engineering, and compromised users, which traditional cybersecurity systems cannot detect.
SentinelMind is a smart AI system that protects computers and systems by watching the person using them, not just the computer itself.

Normally, security systems check:

passwords

IP address

files and network activity

But this project checks the human.

🤔 Why is this needed?

Sometimes:

A real user is forced to do something

An insider tries to misuse access

A person is stressed, scared, or behaving unusually

Hackers trick users (social engineering)

Normal security systems cannot detect this.

👉 SentinelMind can.

In [ ]:
# ==========================================================
# CCDS v4.3 — PhD Level Cognitive Cyber Defense System
# Webcam + Mic + 3D Interactive Forensics + Face Guidance
# ==========================================================
!pip install opencv-python mediapipe librosa sounddevice matplotlib tensorflow scikit-learn
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bunzip2 shape_predictor_68_face_landmarks.dat.bz2
!apt-get update && apt-get install -y portaudio19-dev

import cv2, dlib, time, random
import numpy as np
import librosa
import gradio as gr
import plotly.graph_objects as go

from sklearn.ensemble import IsolationForest
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.layers import LSTM, Dense, TimeDistributed

# ==========================================================
# DLIB SETUP
# ==========================================================

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# ==========================================================S
# GLOBAL STORAGE
# ==========================================================

risk_history = [] # Changed to store lists of [emo, mic, voi, beh, total]
micro_history = []
forensics_log = []

# ==========================================================
# BEHAVIORAL ATTACK ENGINE
# ==========================================================

class AttackPredictionEngine:
    def __init__(self):
        self.model = IsolationForest(contamination=0.25)
        self.model.fit(np.random.rand(1000, 3))

    def predict(self):
        typing = random.uniform(5, 30)
        mouse = random.uniform(0.5, 1.0)
        command = random.uniform(5, 25)
        x = np.array([[typing, mouse, command]])
        return 1 if self.model.predict(x)[0] == -1 else 0

attack_engine = AttackPredictionEngine()

# ==========================================================
# CNN + LSTM EMOTION MODEL
# ==========================================================

class EmotionCNNLSTM:
    def __init__(self):
        self.model = self.build()

    def build(self):
        model = Sequential([
            TimeDistributed(Conv2D(32,(3,3),activation="relu"),
                            input_shape=(5,48,48,1)),
            TimeDistributed(MaxPooling2D()),
            TimeDistributed(Flatten()),
            LSTM(64),
            Dense(1, activation="sigmoid")
        ])
        model.compile(optimizer="adam", loss="binary_crossentropy")
        return model

    def predict(self, frames):
        frames = np.array(frames).reshape(1,5,48,48,1)
        return float(self.model.predict(frames, verbose=0)[0][0])

emotion_model = EmotionCNNLSTM()

# ==========================================================
# FACE GUIDANCE OVERLAY
# ==========================================================

def draw_face_guidance(frame):
    h, w, _ = frame.shape
    cv2.rectangle(frame, (int(w*0.3), int(h*0.2)),
                  (int(w*0.7), int(h*0.8)), (0,255,0), 2)
    cv2.putText(frame, "Align Face Here",
                (int(w*0.35), int(h*0.18)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    return frame

# ==========================================================
# MICRO-EXPRESSION (DLIB)
# ==========================================================

def micro_expression(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)
    if len(faces) == 0:
        return 0.0

    shape = predictor(gray, faces[0])
    pts = np.array([[p.x,p.y] for p in shape.parts()])

    brow = np.linalg.norm(pts[19]-pts[37])
    eye  = np.linalg.norm(pts[41]-pts[37])
    lip  = np.linalg.norm(pts[62]-pts[66])

    return min((brow+eye+lip)/180,1.0)

# ==========================================================
# VOICE STRESS
# ==========================================================

def voice_stress(audio):
    if audio is None:
        return 0.0
    y, sr = librosa.load(audio, sr=22050)
    mfcc = librosa.feature.mfcc(y=y, sr=sr)
    return min(np.mean(np.abs(mfcc))/120,1.0)

# ==========================================================
# 3D VISUALIZATIONS
# ==========================================================

def radar_3d(values):
    labels = ["Emotion","Micro","Voice","Behavior"]
    fig = go.Figure(
        data=[go.Scatterpolar(
            r=values + [values[0]],
            theta=labels + [labels[0]],
            fill='toself'
        )]
    )
    fig.update_layout(
        polar=dict(radialaxis=dict(range=[0,1])),
        title="3D Threat Radar"
    )
    return fig

def heatmap_3d(risk_matrix):
    if len(risk_matrix) < 2:
        # create dummy surface to avoid empty plot
        z = np.zeros((2, 5))
    else:
        z = np.array(risk_matrix)

    x = np.arange(z.shape[1])  # risk dimensions
    y = np.arange(z.shape[0])  # time axis

    fig = go.Figure(data=[go.Surface(
        x=x,
        y=y,
        z=z,
        colorscale="Hot"
    )])

    fig.update_layout(
        title="3D Risk Heatmap (Time × Risk Dimensions)",
        scene=dict(
            xaxis=dict(
                title="Risk Type",
                tickvals=[0,1,2,3,4],
                ticktext=["Emotion","Micro","Voice","Behavior","Total"]
            ),
            yaxis=dict(title="Time"),
            zaxis=dict(title="Risk Intensity", range=[0,1])
        )
    )

    return fig

def replay_3d():
    t = list(range(len(forensics_log)))
    z = [x["risk"] for x in forensics_log]
    fig = go.Figure(data=[go.Scatter3d(
        x=t, y=z, z=z,
        mode="lines+markers"
    )])
    fig.update_layout(
        title="3D Attack Replay Timeline",
        scene=dict(zaxis=dict(range=[0,1]))
    )
    return fig

def micro_expression_graph():
    fig = go.Figure(
        data=[go.Scatter(
            y=micro_history,
            mode="lines+markers"
        )]
    )
    fig.update_layout(
        title="Live Micro-Expression Intensity",
        yaxis=dict(range=[0,1])
    )
    return fig

# ==========================================================
# MAIN PIPELINE
# ==========================================================

def run_ccds(frame, audio):
    frame = draw_face_guidance(frame)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    face = cv2.resize(gray,(48,48))/255.0
    frames = [face]*5

    emo = emotion_model.predict(frames)
    mic = micro_expression(frame)
    voi = voice_stress(audio)

    raw_beh = attack_engine.predict()
    beh = 0.7 if raw_beh == 1 else 0.2  # FIXED

    total = round(max(emo,mic,voi,beh),2)
    # Append the current risk values as a list to risk_history
    risk_history.append([emo, mic, voi, beh, total])
    # Keep only the last 20 entries for heatmap visualization
    risk_history[:] = risk_history[-20:]

    micro_history.append(mic)
    micro_history[:] = micro_history[-30:]

    level = ("🔴 CRITICAL" if total>0.8 else
             "🟠 HIGH" if total>0.6 else
             "🟡 MEDIUM" if total>0.4 else
             "🟢 LOW")

    forensics_log.append({"time":time.time(),"risk":total})

    explanation = f"""
Emotion Risk: {emo:.2f}
Micro-Expression: {mic:.2f}
Voice Stress: {voi:.2f}
Behavior Anomaly: {beh:.2f}

TOTAL RISK: {total}
THREAT LEVEL: {level}
"""

    return (
        frame, emo, mic, voi, beh, total, level, explanation,
        radar_3d([emo,mic,voi,beh]),
        heatmap_3d(risk_history), # Pass the trimmed risk_history to heatmap_3d
        replay_3d(),
        micro_expression_graph()
    )

# ==========================================================
# GRADIO APP
# ==========================================================

with gr.Blocks(title="CCDS v4.3") as demo:
    gr.Markdown("## 🧠 Cognitive Cyber Defense System (PhD Level)")
    gr.Markdown("Human-Centric Zero Trust • 3D Forensics • AI")

    with gr.Row():
        cam = gr.Image(sources=["webcam"], label="Webcam")
        mic = gr.Audio(sources=["microphone"], type="filepath", label="Microphone")

    emo = gr.Number(label="Emotion Risk")
    micr = gr.Number(label="Micro-Expression Risk")
    voi = gr.Number(label="Voice Stress")
    beh = gr.Number(label="Behavior Anomaly")
    total = gr.Number(label="Total Risk")
    level = gr.Textbox(label="Threat Level")
    exp = gr.Textbox(label="Explanation", lines=6)

    radar = gr.Plot(label="3D Threat Radar")
    heat = gr.Plot(label="3D Risk Heatmap")
    replay = gr.Plot(label="3D Attack Replay")
    micro_plot = gr.Plot(label="Live Micro-Expression Graph")

    btn = gr.Button("▶ RUN DEFENSE SCAN")

    btn.click(
        run_ccds,
        inputs=[cam, mic],
        outputs=[cam, emo, micr, voi, beh, total, level, exp,
                 radar, heat, replay, micro_plot]
    )

demo.launch()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b1f93420d8e0463d2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
